# Notebook 4: Tracking con ByteTrack y Demo de Monitoreo EPP

**Trabajo Práctico – Visión por Computadora II**  

---

## Objetivos
1. Integrar **ByteTrack** para seguimiento persistente de trabajadores
2. Asignar ID único a cada persona y mantener su historial de cumplimiento EPP
3. Generar alertas cuando se detecte un trabajador sin EPP reglamentario
4. Procesar un video de obra y guardar el resultado anotado

---

## Arquitectura del Sistema

```
Video de entrada
      │
      ▼
  YOLOv8/v11 ──── Detección de EPP y Personas
      │
      ▼
  ByteTrack ─────── Asigna Track ID persistente por persona
      │
      ▼
  Compliance Engine ── Evalúa EPP por Track ID (casco, chaleco, guantes)
      │
      ▼
  Alert Generator ──── Alerta si no EPP por N frames consecutivos
      │
      ▼
  Video anotado + Reporte de cumplimiento
```

## 1. Configuración

In [ ]:
import json
import cv2
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 120

# Cargar configuración
with open("../data/eval_results.json") as f:
    eval_results = json.load(f)

with open("../data/dataset_config.json") as f:
    ds_config = json.load(f)

CLASS_NAMES = ds_config["class_names"]
MODELS_DIR  = Path("../models")
VIDEOS_DIR  = Path("../videos")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Seleccionar el mejor modelo
best_model_name = max(eval_results, key=lambda m: eval_results[m].get('map50', 0))
model_file_map = {
    'YOLOv8n':  MODELS_DIR / 'yolov8n_best.pt',
    'YOLOv8m':  MODELS_DIR / 'yolov8m_best.pt',
    'YOLOv11n': MODELS_DIR / 'yolov11n_best.pt',
}
BEST_MODEL_PATH = model_file_map[best_model_name]

print(f"Dispositivo:   {DEVICE}")
print(f"Mejor modelo:  {best_model_name}")
print(f"Clases:        {CLASS_NAMES}")

## 2. Motor de Compliance EPP

Define qué clases representan EPP presente y cuáles representan incumplimiento.

In [ ]:
# Clases de EPP y violaciones
EPP_CLASSES = {
    'present': {'Hardhat', 'Mask', 'Safety Vest'},
    'absent':  {'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest'},
}

# Colores de visualización
COLORS = {
    'compliant':     (50,  205, 50),    # verde – trabajador con todo el EPP
    'non_compliant': (220, 50,  50),    # rojo  – trabajador sin EPP
    'unknown':       (180, 180, 180),   # gris  – sin determinar aún
    'alert_bg':      (40,  0,   0),     # fondo alerta
}

class PPEComplianceTracker:
    """
    Mantiene el estado de cumplimiento EPP por Track ID.
    Usa una ventana deslizante de N frames para determinar
    si un trabajador está incumpliendo de forma persistente.
    """
    def __init__(self, window_size: int = 30, alert_threshold: float = 0.7):
        """
        Args:
            window_size:     frames a analizar para determinar estado
            alert_threshold: fracción de frames sin EPP para activar alerta
        """
        self.window_size = window_size
        self.alert_threshold = alert_threshold
        # {track_id: deque of bool (True = cumple, False = no cumple)}
        self.history = defaultdict(lambda: deque(maxlen=window_size))
        self.violations = defaultdict(set)   # {track_id: {classes faltantes}}
        self.total_alerts = 0
        self.alerted_ids = set()

    def update(self, track_id: int, detected_classes: list) -> dict:
        """
        Actualiza el estado de un trabajador dado su track_id
        y las clases detectadas en el frame actual.
        
        Returns: dict con status y missing_ppe
        """
        missing = EPP_CLASSES['absent'].intersection(set(detected_classes))
        compliant = len(missing) == 0
        
        self.history[track_id].append(compliant)
        self.violations[track_id] = missing
        
        # Determinar status basado en ventana
        if len(self.history[track_id]) < self.window_size // 3:
            status = 'unknown'
        else:
            non_compliant_ratio = 1 - (sum(self.history[track_id]) / len(self.history[track_id]))
            if non_compliant_ratio >= self.alert_threshold:
                status = 'non_compliant'
                if track_id not in self.alerted_ids:
                    self.total_alerts += 1
                    self.alerted_ids.add(track_id)
            else:
                status = 'compliant'
                self.alerted_ids.discard(track_id)  # si mejora, resetear alerta
        
        return {'status': status, 'missing_ppe': missing}

    def get_compliance_rate(self) -> float:
        """Porcentaje global de cumplimiento en el video."""
        if not self.history:
            return 1.0
        all_frames = [v for hist in self.history.values() for v in hist]
        return sum(all_frames) / len(all_frames) if all_frames else 1.0

print("✅ PPEComplianceTracker definido")

## 3. Pipeline de Procesamiento de Video con ByteTrack

In [ ]:
def draw_compliance_overlay(frame: np.ndarray, track_id: int, bbox, 
                             status: str, missing_ppe: set, conf: float) -> np.ndarray:
    """
    Dibuja el bounding box de la persona con:
    - Color según estado de cumplimiento
    - Track ID
    - Lista de EPP faltante (si aplica)
    - Barra de confianza
    """
    x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    color = COLORS[status]
    
    # Bounding box principal
    thickness = 3 if status == 'non_compliant' else 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    
    # Etiqueta con Track ID
    label = f"ID:{track_id}  {conf:.0%}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    
    # EPP faltante (alerta)
    if missing_ppe:
        y_offset = y1 + 20
        for item in missing_ppe:
            alert_text = f"⚠ {item}"
            cv2.putText(frame, alert_text, (x1 + 5, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
            y_offset += 18
    
    return frame


def add_stats_overlay(frame: np.ndarray, frame_num: int, total_tracks: int,
                       compliant: int, non_compliant: int, compliance_rate: float) -> np.ndarray:
    """Panel de estadísticas en la esquina superior izquierda."""
    h, w = frame.shape[:2]
    overlay = frame.copy()
    
    # Fondo semitransparente
    cv2.rectangle(overlay, (10, 10), (280, 120), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    
    stats = [
        f"Frame: {frame_num:04d}",
        f"Trabajadores: {total_tracks}",
        f"Cumplen EPP: {compliant}",
        f"Incumplen:   {non_compliant}",
        f"Compliance:  {compliance_rate:.1%}",
    ]
    
    for i, text in enumerate(stats):
        color = (0, 255, 0) if i < 2 else ((0, 200, 0) if i == 2 else (0, 0, 255) if i == 3 else (255, 255, 255))
        cv2.putText(frame, text, (18, 32 + i * 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Alerta grande si hay incumplimiento
    if non_compliant > 0:
        alert = f"ALERTA: {non_compliant} trabajador(es) sin EPP!"
        (aw, ah), _ = cv2.getTextSize(alert, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
        cv2.rectangle(frame, (w//2 - aw//2 - 10, 5), (w//2 + aw//2 + 10, 35), (0, 0, 180), -1)
        cv2.putText(frame, alert, (w//2 - aw//2, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
    
    return frame


print("✅ Funciones de visualización definidas")

In [ ]:
def process_video_with_tracking(
    input_video: str,
    output_video: str,
    model_path: str,
    conf_threshold: float = 0.4,
    max_frames: int = None,  # None = todo el video
) -> dict:
    """
    Procesa un video aplicando detección YOLO + ByteTrack.
    
    Returns: dict con estadísticas del video procesado.
    """
    model = YOLO(model_path)
    tracker = PPEComplianceTracker(window_size=30, alert_threshold=0.6)
    
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {input_video}")
    
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video: {width}×{height} @ {fps:.1f} FPS  |  {total} frames")
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
    
    frame_num = 0
    person_class_id = None
    
    # Encontrar el ID de la clase 'Person'
    if 'Person' in CLASS_NAMES:
        person_class_id = CLASS_NAMES.index('Person')
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_num >= max_frames:
            break
        
        # Detección + ByteTrack en un solo paso
        results = model.track(
            frame,
            persist    = True,          # mantener tracks entre frames
            tracker    = 'bytetrack.yaml',
            conf       = conf_threshold,
            device     = DEVICE,
            verbose    = False,
            classes    = None,          # detectar todas las clases
        )
        
        annotated_frame = frame.copy()
        current_status = {}
        
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes    = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
            confs    = results[0].boxes.conf.cpu().numpy()
            
            # Agrupar detecciones por track_id
            track_detections = defaultdict(list)
            track_boxes = {}
            track_confs = {}
            
            for bbox, tid, cid, conf in zip(boxes, track_ids, class_ids, confs):
                cls_name = CLASS_NAMES[cid]
                track_detections[tid].append(cls_name)
                # Guardar bbox con mayor confianza para cada track
                if tid not in track_boxes or conf > track_confs[tid]:
                    track_boxes[tid] = bbox
                    track_confs[tid] = conf
            
            # Actualizar tracker de compliance y dibujar
            compliant_count = non_compliant_count = 0
            
            for tid, detected_cls in track_detections.items():
                # Solo procesar si hay una detección de 'Person'
                if 'Person' not in detected_cls:
                    continue
                
                result = tracker.update(tid, detected_cls)
                status  = result['status']
                missing = result['missing_ppe']
                current_status[tid] = status
                
                if status == 'compliant':     compliant_count += 1
                elif status == 'non_compliant': non_compliant_count += 1
                
                if tid in track_boxes:
                    draw_compliance_overlay(
                        annotated_frame, tid, track_boxes[tid],
                        status, missing, track_confs[tid]
                    )
            
            # Overlay de estadísticas
            annotated_frame = add_stats_overlay(
                annotated_frame, frame_num,
                len(track_detections),
                compliant_count, non_compliant_count,
                tracker.get_compliance_rate()
            )
        
        out.write(annotated_frame)
        frame_num += 1
        
        if frame_num % 50 == 0:
            print(f"  Procesado: {frame_num}/{total if total else '?'} frames"
                  f"  | Tracks activos: {len(current_status)}")
    
    cap.release()
    out.release()
    
    stats = {
        'frames_procesados': frame_num,
        'tracks_unicos': len(tracker.history),
        'total_alertas': tracker.total_alerts,
        'compliance_rate': round(tracker.get_compliance_rate(), 4),
        'output_video': output_video,
    }
    print(f"\n✅ Video guardado: {output_video}")
    print(f"   Frames:         {stats['frames_procesados']}")
    print(f"   Tracks únicos:  {stats['tracks_unicos']}")
    print(f"   Alertas:        {stats['total_alertas']}")
    print(f"   Compliance:     {stats['compliance_rate']:.1%}")
    return stats

print("✅ Pipeline de tracking definido")

## 4. Ejecutar Demo

### Opción A: Con video propio
Colocar un video de obra en `videos/input.mp4` y ejecutar la siguiente celda.

### Opción B: Descargar video de muestra

In [ ]:
# ==========================================================
# Opción B: Descargar video de construcción de YouTube
# Requiere: pip install yt-dlp
# ==========================================================
import subprocess
import os

input_video = str(VIDEOS_DIR / "input.mp4")

if not os.path.exists(input_video):
    print("Descargando video de muestra (requiere yt-dlp)...")
    # Video de obra de construcción – dominio público
    url = "https://www.youtube.com/watch?v=DLknkTG1EpE"
    result = subprocess.run([
        "yt-dlp",
        "-f", "best[height<=480]",
        "-o", input_video,
        "--max-filesize", "50m",
        url
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Video descargado: {input_video}")
    else:
        print(f"⚠️  Error descargando video: {result.stderr}")
        print("   Alternativa: colocá manualmente un video en videos/input.mp4")
else:
    print(f"✅ Video de entrada encontrado: {input_video}")

In [ ]:
# ==============================
# EJECUTAR EL DEMO
# ==============================
output_video = str(VIDEOS_DIR / "output_tracked.mp4")

if not Path(input_video).exists():
    print("⚠️  No hay video de entrada. Colocá un video en videos/input.mp4")
else:
    stats = process_video_with_tracking(
        input_video    = input_video,
        output_video   = output_video,
        model_path     = str(BEST_MODEL_PATH),
        conf_threshold = 0.4,
        max_frames     = 300,   # procesar solo 300 frames para la demo
    )
    
    # Guardar estadísticas
    with open("../data/demo_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print("\nEstadísticas guardadas en data/demo_stats.json")

## 5. Preview del Video Procesado

In [ ]:
def extract_video_frames(video_path: str, n_frames: int = 6, 
                          start_ratio: float = 0.1) -> list:
    """Extrae N frames distribuidos del video para visualización."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(int(total * start_ratio), total - 1, n_frames, dtype=int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


if Path(output_video).exists():
    frames = extract_video_frames(output_video, n_frames=6)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, (ax, frame) in enumerate(zip(axes, frames)):
        ax.imshow(frame)
        ax.axis('off')
        ax.set_title(f"Frame {i+1}", fontsize=10)
    
    plt.suptitle(f"Preview del Video Procesado – {best_model_name} + ByteTrack",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("../data/demo_preview.png", bbox_inches='tight', dpi=150)
    plt.show()
    print(f"\nPreview guardado en data/demo_preview.png")
    print(f"Video completo: {output_video}")
else:
    print("⚠️  Video de salida no encontrado. Ejecutar el demo primero.")

## 6. Reporte Final del Sistema

In [ ]:
# Reporte final integrado
print("=" * 65)
print("  REPORTE FINAL – Sistema de Monitoreo EPP")
print("=" * 65)

# Métricas del mejor modelo
e = eval_results[best_model_name]
print(f"""
MODELO SELECCIONADO: {best_model_name}
─────────────────────────────────────
Métricas de Detección (Construction-PPE Dataset):
  mAP@50:        {e['map50']:.4f}
  mAP@50-95:     {e['map50_95']:.4f}
  Precision:     {e['precision']:.4f}
  Recall:        {e['recall']:.4f}

Componentes del sistema:
  • Detector:    {best_model_name} fine-tuneado en Construction-PPE
  • Tracker:     ByteTrack (Ultralytics)
  • Compliance:  Ventana deslizante de 30 frames / umbral 60%
  • Alertas:     Visual en tiempo real + log por Track ID

Clases monitoreadas:
  EPP Presente:  Hardhat ✓ | Mask ✓ | Safety Vest ✓
  EPP Ausente:   NO-Hardhat ✗ | NO-Mask ✗ | NO-Safety Vest ✗
""")

if Path("../data/demo_stats.json").exists():
    with open("../data/demo_stats.json") as f:
        ds = json.load(f)
    print(f"""Resultado del Demo:
  Frames procesados: {ds['frames_procesados']}
  Trabajadores detectados (tracks únicos): {ds['tracks_unicos']}
  Alertas de incumplimiento: {ds['total_alertas']}
  Tasa de cumplimiento global: {ds['compliance_rate']:.1%}
""")

print("=" * 65)

---
## ✅ Checklist Notebook 4
- [ ] PPEComplianceTracker implementado
- [ ] Pipeline de video con ByteTrack funcionando
- [ ] Video de demo procesado y guardado
- [ ] Preview de frames visualizado
- [ ] Reporte final generado

---
## 🎯 Trabajo Práctico Completado

**Resumen de entregables:**
1. `notebooks/01_setup_and_dataset.ipynb` – EDA del dataset Construction-PPE
2. `notebooks/02_model_training.ipynb` – Fine-tuning de YOLOv8n, YOLOv8m, YOLOv11n
3. `notebooks/03_model_comparison.ipynb` – Comparación de arquitecturas y métricas
4. `notebooks/04_tracking_demo.ipynb` – Demo con ByteTrack y sistema de alertas